# Phase 3B: Grad-CAM Visual Audit Report (Re-Audit)

## Summary Verdict
**NO_GO_FOR_PHASE_4_UNTIL_REAL_LOCALIZATION_AUDIT**.

**Reasoning:** RSNA Pneumonia Detection data is now loaded from `data/rsna/`, and the Phase 3 grounding evaluator generated real-image Grad-CAM overlay artifacts under `results/overlays/rsna/`. The current classifier checkpoint is still smoke-trained, so the generated report keeps `WARNING_DO_NOT_USE` and `model_quality_evidence: false`. Diagnostic localization quality remains unverified. Owner direction supersedes the earlier `CONDITIONAL_GO`: Phase 4 is blocked until Gemini audits the real RSNA overlays and the owner explicitly accepts the remaining smoke-checkpoint limitation or provides a real trained checkpoint.

## Dataset/Sample Coverage
- **Original Target:** VinDr-CXR (Unavailable).
- **Alternative Dataset:** RSNA Pneumonia Detection Dataset (`data/rsna/`).
- **RSNA Mapping:** NIH `Pneumonia` maps to RSNA `Lung Opacity`; all other NIH labels are intentionally unmapped.
- **Current RSNA Run:** `results/grounding_rsna_eval.json` evaluates 1,181 real RSNA validation images with 1,880 ground-truth opacity boxes. It generated 28 confidence-gated CAMs and saved 20 overlay PNGs plus `results/overlays/rsna/rsna_grid.png`.

## Visual Audit Checklist (Scorecard)
*Pending Gemini review of generated real RSNA overlays. Criteria specifically adjusted for RSNA features:*

1. **Is the model attending to the correct anatomical region?**
   - *Pending Gemini audit.* Current smoke-checkpoint mAP@0.5 and pointing-game accuracy are 0.0, so no localization-quality claim is allowed.
2. **Does CAM focus inside lung fields?**
   - *Pending Gemini audit.* RSNA bounding boxes localize lung opacities, but the current checkpoint was not trained on real NIH/RSNA signal.
3. **Does CAM fire on image borders?**
   - *Pending Gemini audit.* Use Codex's `heatmap_border_fraction` hook in the generated report.
4. **Does CAM fire on L/R anatomical markers?**
   - *Pending Gemini audit.* A real audit must verify if models spuriously use text markers.
5. **Does CAM fire on pacemakers, tubes, labels, text, or artifacts?**
   - *Pending Gemini audit.* RSNA may contain chest tubes; manually audit for spurious correlation.
6. **Does CAM fire outside the lung field?**
   - *Pending Gemini audit.* Sometimes the models focus on shoulders/clavicles in real datasets.
7. **Are false positives visually suspicious?**
   - *Pending Gemini audit.*
8. **Are false negatives explained by weak/noisy activation?**
   - *Pending Gemini audit.* Predictions failing the confidence gate still correctly yield no spatial heatmaps.
9. **Which findings appear to have more reliable localization?**
   - *Expected for RSNA:* Large Lung Opacities should be easier than subtle opacities.
10. **Which findings should not be trusted visually?**
   - *Expected for RSNA:* Subtle opacities and areas close to the diaphragm.

## L/R Marker and Border Artifact Check
- Thanks to Codex's hardening, border artifact checks are now a metric (`heatmap_border_fraction` and `heatmap_peak_in_border`). These remain crucial for the real data sweep.

## Safety Concerns
- **Clinical Claim Prevention:** The overlaid visualizations come from a smoke-trained checkpoint. The pipeline is safe only because of `WARNING_DO_NOT_USE`, the explicit NOT A CLINICAL EVALUATION banners, and the confidence gate.
- **Grad-CAM Resolution:** Grad-CAM from a DenseNet-121 final convolutional layer has a relatively coarse spatial resolution ($7 \\times 7$), which poses a safety risk if users trust it for small findings like micro-nodules.

## Required Fixes Before Phase 4
The pipeline architecture is structurally healthy, but Phase 4 remains blocked by unresolved real-data audit and smoke-checkpoint limitations. Required before Phase 4:

1. Gemini visual audit on `results/overlays/rsna/rsna_grid.png` and the 20 individual RSNA overlay PNGs.
2. Owner / Claude decision on whether the smoke-checkpoint RSNA audit is enough for Phase 4 engineering continuation or whether a real trained classifier checkpoint is required first.
3. If required, train or fine-tune a real classifier checkpoint, then rerun `make eval-grounding-rsna`.

## Recommended Next Action
1. **Gemini:** Audit `results/overlays/rsna/rsna_grid.png` and `results/grounding_rsna_eval.json`.
2. **Owner / Claude:** Resolve the smoke-checkpoint limitation before Phase 4.